---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"
---

In [1]:
knitr::opts_chunk$set(echo = TRUE)


## Load Required Libraries

In [2]:
options(verbose = FALSE)
options(warn = -1)


In [3]:
# suppressPackageStartupMessages({
# library(reticulate)
# library(tidyverse)
library(data.table)
library(here)
library(tictoc)
library(stringr)
library(stringi)
library(lubridate)
library(docstring)
library(profvis)
library(hash)
# library(foreach)
# library(doParallel)
# library(parallel)
library(future)
library(future.apply)
library(knitr)
# })


In [4]:
options(warn = 1)


## Set Parameters

### Year to Load & Version

In [5]:
year_to_load <- "2018"
version <- "v2"


### Parameters

In [6]:
sample_size <- 25 * 1e3
seed <- 123

drop_cols <- c(paste0("ICDCODE", 13:14), "ICCODED15", paste0("ICDCODE", 16:170))
icd_cols <- paste0("clin_icd", 1:12)
rvs_cols <- paste0("clin_rvs", 1:20)


In [7]:
to_read <- FALSE
to_sample <- TRUE
to_write <- TRUE
to_group <- TRUE
to_filter <- FALSE # unused
to_profvis <- FALSE
to_chunk <- FALSE
to_view_checks <- TRUE


In [8]:
set.seed(seed)
options(future.globals.maxSize = 1024 * 1024^2)
global_seed <- seed # for parallelized operations


## Source Data Formats

In [9]:
source(here("data-cleaning", "r_scripts", "data-formats.R"))
source(here("data-cleaning", "r_scripts", "file-paths.R"))


## Source Functions

In [10]:
source(here("data-cleaning", "r_scripts", "general-functions.R"))
source(here("data-cleaning", "r_scripts", "main-functions.R"))
source(here("data-cleaning", "r_scripts", "icd-functions.R"))
source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("data-cleaning", "r_scripts", "grouper-functions.R"))


## Load Mapping Data

In [11]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read Data

### Reading Data

In [12]:
options(verbose = FALSE)
options(warn = -1)

dt <- main_read_function()

if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
} else {
  total_rows <- fread(full_claims_file, select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file)
}


In [13]:
options(warn = 1)


## Data Processing

### Data Cleaning

#### Not chunking

In [14]:
if (!to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    profvis({
      dt <- clean_data(dt)
    })
  } else {
    dt <- clean_data(dt)
  }
}


#### Chunking

In [15]:
if (to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    profvis({
      num_cores <- max(1, availableCores() - 1)
      chunk_size <- ceiling(nrow(dt) / num_cores)
      chunks <- split(dt, rep(1:num_cores,
        each = chunk_size,
        length.out = nrow(dt)
      ))
      # Plan for parallel processing
      plan(multisession, workers = num_cores)
      # Process each chunk in parallel
      processed_chunks <- future_lapply(chunks, process_chunk,
        future.seed = global_seed
      )
      # Combine processed chunks
      dt <- rbindlist(processed_chunks)
      dt <- replace_empty_with_na(dt, to_view_checks)
    })
  } else {
    num_cores <- max(1, availableCores() - 1)
    chunk_size <- ceiling(nrow(dt) / num_cores)
    chunks <- split(dt, rep(1:num_cores,
      each = chunk_size,
      length.out = nrow(dt)
    ))
    # Plan for parallel processing
    plan(multisession, workers = num_cores)
    # Process each chunk in parallel
    processed_chunks <- future_lapply(chunks, process_chunk,
      future.seed = global_seed
    )
    # Combine processed chunks
    dt <- rbindlist(processed_chunks)
    dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


### Map codes and Find PDx (if not chunking)

#### Map Codes

In [16]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    profvis({
      dt <- map_rvs_icd9(dt, rvs_icd9)
      dt <- implement_icd10_mapping(dt)
      # Replace empty strings in character and factor columns with NA
      # dt <- replace_empty_with_na(dt, to_view_checks)
    })
  } else {
    dt <- map_rvs_icd9(dt, rvs_icd9)
    dt <- implement_icd10_mapping(dt)
    # Replace empty strings in character and factor columns with NA
    # dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


#### Find PDx

In [17]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    profvis({
      # dt <- apply_find_pdx(dt)
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
  } else {
    # dt <- apply_find_pdx(dt)
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}


In [18]:
if (to_write) {
  fwrite(dt, here(path_to_intermediate, paste0(
    "output_", year_to_load,
    suffix, ".csv"
  )))
}


## Export

### Export for Batch Grouper

In [19]:
if (to_group) {
  if (to_profvis) {
    profvis({
      export_for_batch_grouper(dt, year_to_load, output_txt_file)
      for_batch_grouping <- fread(output_txt_file,
        sep = "|", na.strings = "--"
      )
      batch_grouping_result <- fread(grouper_result_file,
        sep = "|", na.strings = "--"
      )
    })
  } else {
    export_for_batch_grouper(dt, year_to_load, output_txt_file)
    for_batch_grouping <- fread(output_txt_file,
      sep = "|", na.strings = "--"
    )
    batch_grouping_result <- fread(grouper_result_file,
      sep = "|", na.strings = "--"
    )
  }
}


## Runtime Estimation

### Stop Timer

In [20]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


### Calculate Speed

In [21]:
# Calculate time spent per cell and per row
total_rows_dt <- nrow(dt)
total_cells <- nrow(dt) * ncol(dt)

time_per_cell <- total_time / total_cells
time_per_row <- total_time / total_rows_dt
time_estimate_total_rows <- time_per_row * total_rows

# Format the row numbers
formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
formatted_total_rows <- format_large_numbers(total_rows)

# Print the results with aligned decimal points and formatted row numbers
cat(sprintf(
  "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
  formatted_total_rows_dt, total_time
))
cat(sprintf(
  "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
  formatted_total_rows_dt, time_per_row * 1000
))
cat(sprintf(
  "Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
  formatted_total_rows, time_estimate_total_rows / 60
))


### Old Python Code

In [ ]:
# # Import pandas
# pandas <- import("pandas")

# # Step 1: Copy the master_dt to avoid modifying the original data
# python_input_dt <- fread(here(checkpoint_6_path, paste0(checkpoint_6_prefix, ".csv")))

# # Convert columns to Date objects, ignoring NA values
# date_columns <- names(python_input_dt)[grepl("date", names(python_input_dt))]

# python_input_dt[, pat_bdate_orig := as.Date(pat_bdate, format = "%m/%d/%Y")]
# python_input_dt[, (date_columns) := lapply(.SD, as.Date, format = "%m/%d/%Y"), .SDcols = date_columns]

# # Create a data.table for rows with negative `pat_age`
# negative_age_dt <- python_input_dt[pat_age < 0, ]

# # Create a data.table for rows with non-negative `pat_age`
# positive_age_dt <- python_input_dt[pat_age >= 0, ]

# # Process rows with non-negative `pat_age`
# positive_age_dt[, pat_bdate := dmy(generate_dob(format(pat_bdate_orig, "%Y-%m-%d"), pat_age, format(date_adm, "%Y-%m-%d")))]

# # For negative `pat_age`, keep `pat_bdate` as it is
# negative_age_dt[, pat_bdate := pat_bdate_orig]

# # Combine the processed data
# result_dt <- rbind(positive_age_dt, negative_age_dt)

# # Write the result to CSV
# csv_path <- here(checkpoint_7_path, paste0(checkpoint_7a_prefix, ".csv"))

# fwrite(result_dt, csv_path)

# # Use reticulate to run the following Python code within the R environment
# py_run_string(paste0("
# import pandas as pd

# # Create a dictionary with all columns set to 'string' type
# dtype_dict = {
#     'id_series': 'string',
#     'id_pin': 'string',
#     'date_adm': 'string',
#     'time_adm': 'string',
#     'date_dis': 'string',
#     'time_dis': 'string',
#     'date_rec': 'string',
#     'date_ref': 'string',
#     'date_check': 'string',
#     'id_hci': 'string',
#     'id_hcp': 'string',
#     'clin_outpatient': 'string',
#     'clin_emergency': 'string',
#     'pat_type': 'category',
#     'clin_acc': 'category',
#     'pat_rel': 'category',
#     'pat_bdate': 'string',
#     'pat_age': 'float64',
#     'pat_sex': 'category',
#     'pat_bwt': 'float64',
#     'pat_memcat_parent': 'category',
#     'pat_memcat_child': 'category',
#     'clin_discharge': 'Int64',
#     'clin_c1': 'string',
#     'clin_c2': 'string',
#     'claim_status': 'category',
#     'claim_payout': 'float64',
#     'claim_charge': 'float64',
#     'date_ext': 'string',
#     'id_year': 'string',
#     'clin_icd': 'string',
#     'clin_rvs': 'string',
#     'clin_c1_orig': 'string',
#     'clin_c2_orig': 'string',
#     'icd9_list': 'string',
#     'pdx': 'string',
#     'pdx_code': 'string',
#     'pat_bdate_orig': 'string'
# }

# # Read the CSV with the specified dtype
# pandas_df = pd.read_csv('", csv_path, "', dtype=dtype_dict)
# "))

# # Access the pandas DataFrame in R if needed
# pandas_df <- py$pandas_df

# # View the structure
# str(pandas_df)

# # Step 3: Convert `clin_icd` and `icd9_list` columns, replace NaN with "None"
# py_run_file(here("data-cleaning", "py_scripts", "format_data.py"))

# # Retrieve the processed DataFrame back to R
# subset_df <- py$output

# # str(subset_df)
# # Step 1: Ensure date columns are properly converted to Date objects
# subset_df$date_adm <- as.Date(subset_df$date_adm, format = "%Y-%m-%d")
# subset_df$pat_bdate <- as.Date(subset_df$pat_bdate, format = "%Y-%m-%d")

# # Step 2: Initialize the 'ageday' column with NA
# subset_df$ageday <- NA

# # Step 3: Calculate 'ageday' only for valid rows where pat_age == 0
# valid_rows <- subset_df$patage >= 0 & subset_df$patage %% 1 == 0

# subset_df$ageday[valid_rows & subset_df$patage == 0] <- as.numeric(
#   difftime(
#     subset_df$date_adm[valid_rows & subset_df$patage == 0],
#     subset_df$pat_bdate[valid_rows & subset_df$patage == 0],
#     units = "days"
#   )
# )

# subset_df$date_adm <- as.POSIXct(paste(subset_df$date_adm, subset_df$time_adm), format = "%Y-%m-%d %H:%M:%S")
# subset_df$date_dis <- as.POSIXct(paste(subset_df$date_dis, subset_df$time_dis), format = "%Y-%m-%d %H:%M:%S")

# # Step 4: Replace NA values in 'ageday' with "None"
# subset_df$ageday[is.na(subset_df$ageday)] <- "None"

# fwrite(as.data.table(subset_df), here(checkpoint_7_path, paste0(checkpoint_7b_prefix, ".csv")))
# # Step 5: Convert the DataFrame to a format suitable for Python processing
# py$pandas_df <- pandas$read_csv(here(checkpoint_7_path, paste0(checkpoint_7b_prefix, ".csv")))

# # Step 6: Process each row of the DataFrame through `drg_seeker` and append results
# py_run_file(here("data-cleaning", "py_scripts", "run_drg_seeker.py"))
